# Skin Cancer Detection - Training Model
This notebook trains a CNN on the ISIC 2018 dataset to classify 7 types of skin lesions.

In [ ]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# 1. Load GroundTruth
csv_path = r'E:\c programming\MLproject\archive\GroundTruth.csv'
image_dir = r'E:\c programming\MLproject\archive\images'

df = pd.read_csv(csv_path)
# The image IDs don't have .jpg in the CSV, let's append it
df['image'] = df['image'] + '.jpg'

print("Total images in dataset:", len(df))
df.head()

In [ ]:
# 2. Data Generators
# The labels are one-hot encoded: MEL, NV, BCC, AKIEC, BKL, DF, VASC
classes = ['MEL', 'NV', 'BCC', 'AKIEC', 'BKL', 'DF', 'VASC']

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2 # 80% train, 20% validation
)

train_generator = datagen.flow_from_dataframe(
    dataframe=df,
    directory=image_dir,
    x_col='image',
    y_col=classes,
    target_size=(64, 64),
    batch_size=32,
    class_mode='raw', # Because y_col is already one-hot encoded
    subset='training'
)

validation_generator = datagen.flow_from_dataframe(
    dataframe=df,
    directory=image_dir,
    x_col='image',
    y_col=classes,
    target_size=(64, 64),
    batch_size=32,
    class_mode='raw',
    subset='validation'
)

In [ ]:
# 3. Model Definition
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(64, 64, 3)),
    MaxPooling2D(2, 2),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(7, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
# 4. Training
print("Starting training...")
history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=10
)

In [ ]:
# 5. Save Model
model.save('skin_cancer_model.keras')
print("Model saved to skin_cancer_model.keras")

In [ ]:
import matplotlib.pyplot as plt
import json

os.makedirs('static', exist_ok=True)

# Plot accuracy and loss
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['accuracy'], label='Train Accuracy')
ax1.plot(history.history['val_accuracy'], label='Val Accuracy')
ax1.set_title('Model Accuracy')
ax1.set_ylabel('Accuracy')
ax1.set_xlabel('Epoch')
ax1.legend()

ax2.plot(history.history['loss'], label='Train Loss')
ax2.plot(history.history['val_loss'], label='Val Loss')
ax2.set_title('Model Loss')
ax2.set_ylabel('Loss')
ax2.set_xlabel('Epoch')
ax2.legend()

plt.savefig('static/training_history.png')
plt.show()


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import seaborn as sns
import numpy as np

print("Evaluating on validation set...")
validation_generator.reset()
Y_pred = model.predict(validation_generator)
y_pred = np.argmax(Y_pred, axis=1)

# Get true labels
y_true = np.argmax(validation_generator.labels, axis=1)

# Classification Report
report = classification_report(y_true, y_pred, target_names=classes, output_dict=True)
print(classification_report(y_true, y_pred, target_names=classes))

# Confusion Matrix Heatmap
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.savefig('static/confusion_matrix.png')
plt.show()

# Save metrics to JSON for Flask
metrics = {
    'accuracy': accuracy_score(y_true, y_pred),
    'report': report
}
with open('static/metrics.json', 'w') as f:
    json.dump(metrics, f)
print("Metrics and graphs saved to static folder!")
